In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

import matplotlib.pyplot as plt

In [2]:
# Load dataset
file_path = r"C:\Users\maria\Downloads\spaceship_titanic.csv"
df = pd.read_csv(file_path)

# Quick check
print(df.shape)
df.head()

(8693, 14)


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [3]:
# Cabin to Deck
df['Deck'] = df['Cabin'].astype('string').str[0]
valid_decks = set(list("ABCDEFGT"))
df['Deck'] = df['Deck'].where(df['Deck'].isin(valid_decks))

# Spending features
spend_cols = ['RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']
df['TotalSpend'] = df[spend_cols].sum(axis=1)
df['NoSpend'] = (df['TotalSpend'] == 0).astype(int)

# Convert booleans
for col in ['CryoSleep','VIP','Transported']:
    if col in df.columns:
        df[col] = df[col].astype('bool').astype(int)

# Drop identifiers
for col in ['PassengerId','Name','Cabin']:
    if col in df.columns:
        df = df.drop(columns=[col])

# Drop rows with missing values
df = df.dropna()

# Categoricals
cat_cols = [c for c in ['HomePlanet','Destination','Deck'] if c in df.columns]
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

print(df.shape)
df.head()

(7111, 22)


,CryoSleep,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,TotalSpend,...,HomePlanet_Mars,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,Deck_B,Deck_C,Deck_D,Deck_E,Deck_F,Deck_G,Deck_T
0,0,39.0,0,0.0,0.0,0.0,0.0,0.0,0,0.0,...,False,False,True,True,False,False,False,False,False,False
1,0,24.0,0,109.0,9.0,25.0,549.0,44.0,1,736.0,...,False,False,True,False,False,False,False,True,False,False
2,0,58.0,1,43.0,3576.0,0.0,6715.0,49.0,0,10383.0,...,False,False,True,False,False,False,False,False,False,False
3,0,33.0,0,0.0,1283.0,371.0,3329.0,193.0,0,5176.0,...,False,False,True,False,False,False,False,False,False,False
4,0,16.0,0,303.0,70.0,151.0,565.0,2.0,1,1091.0,...,False,False,True,False,False,False,False,True,False,False


In [4]:
X = df.drop(columns=['Transported'])
y = df['Transported'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape

((5688, 21), (1423, 21))

In [5]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier

base_tree = DecisionTreeClassifier(random_state=42)

# Bagging
bagging = BaggingClassifier(
    estimator=base_tree,
    n_estimators=200,
    bootstrap=True,
    n_jobs=-1,
    random_state=42
)
bagging.fit(X_train, y_train)
print("Bagging accuracy:", accuracy_score(y_test, bagging.predict(X_test)))

# Pasting
pasting = BaggingClassifier(
    estimator=base_tree,
    n_estimators=200,
    bootstrap=False,
    n_jobs=-1,
    random_state=42
)
pasting.fit(X_train, y_train)
print("Pasting accuracy:", accuracy_score(y_test, pasting.predict(X_test)))

Bagging accuracy: 0.7842586085734364
Pasting accuracy: 0.7561489810260014


In [6]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_features='sqrt',
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train, y_train)
print("Random Forest accuracy:", accuracy_score(y_test, rf.predict(X_test)))

Random Forest accuracy: 0.7933942375263527


In [7]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train, y_train)
print("Gradient Boosting accuracy:", accuracy_score(y_test, gb.predict(X_test)))

Gradient Boosting accuracy: 0.8025298664792692


In [8]:
from sklearn.ensemble import AdaBoostClassifier

ada = AdaBoostClassifier(
    n_estimators=300,
    learning_rate=0.5,
    random_state=42
)
ada.fit(X_train, y_train)
print("AdaBoost accuracy:", accuracy_score(y_test, ada.predict(X_test)))

AdaBoost accuracy: 0.7786366830639494


In [9]:
# Collect accuracies from all models
results = {
    "Bagging": accuracy_score(y_test, bagging.predict(X_test)),
    "Pasting": accuracy_score(y_test, pasting.predict(X_test)),
    "Random Forest": accuracy_score(y_test, rf.predict(X_test)),
    "Gradient Boosting": accuracy_score(y_test, gb.predict(X_test)),
    "AdaBoost": accuracy_score(y_test, ada.predict(X_test)),
}

# Turn into DataFrame
results_df = pd.DataFrame.from_dict(results, orient='index', columns=['Accuracy'])
results_df = results_df.sort_values(by='Accuracy', ascending=False)

print("Model comparison (higher = better):")
display(results_df)

# Best model
best_model = results_df['Accuracy'].idxmax()
best_acc = results_df['Accuracy'].max()
print(f"\nThe best model is: {best_model} with accuracy = {best_acc:.3f}")

Model comparison (higher = better):


,Accuracy
Gradient Boosting,0.802530
Random Forest,0.793394
Bagging,0.784259
AdaBoost,0.778637
Pasting,0.756149



The best model is: Gradient Boosting with accuracy = 0.803
